In [46]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv(r"E:\summit\task1_data_air\Airbnb_Open_Data.csv")

C:\Users\eyadm\AppData\Local\Temp\ipykernel_1800\788770611.py:1: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"E:\summit\task1_data_air\Airbnb_Open_Data.csv")


In [48]:
df.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  str    
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  str    
 4   host name                       102193 non-null  str    
 5   neighbourhood group             102570 non-null  str    
 6   neighbourhood                   102583 non-null  str    
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  str    
 10  country code                    102468 non-null  str    
 11  instant_bookable                102494 non-null  object 
 12  cancellation_policy        

In [50]:
df.dtypes

id                                  int64
NAME                                  str
host id                             int64
host_identity_verified                str
host name                             str
neighbourhood group                   str
neighbourhood                         str
lat                               float64
long                              float64
country                               str
country code                          str
instant_bookable                   object
cancellation_policy                   str
room type                             str
Construction year                 float64
price                                 str
service fee                           str
minimum nights                    float64
number of reviews                 float64
last review                           str
reviews per month                 float64
review rate number                float64
calculated host listings count    float64
availability 365                  

# data format 

In [51]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='str')

# drop useless columns 

In [52]:
df = df.drop(columns=['license', 'country', 'country_code','host_name'])

# drop dublicates 

In [53]:
print(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)

541
(102058, 22)


# Drop nulls , changeing the Last_review to datetime , removing the $ from price and service fee

In [54]:
for col in ['price', 'service_fee']:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['last_review'] = pd.to_datetime(df["last_review"])

In [55]:
df = df.dropna(subset=['review_rate_number', 'host_identity_verified', 'last_review', 'lat', 'long','instant_bookable']).reset_index(drop=True)
print(df.shape)

(85626, 22)


In [56]:
print(df["price"])
df.dtypes

0         966.0
1         142.0
2         368.0
3         204.0
4         577.0
          ...  
85621    1183.0
85622     696.0
85623     909.0
85624     387.0
85625    1128.0
Name: price, Length: 85626, dtype: float64


id                                         int64
name                                         str
host_id                                    int64
host_identity_verified                       str
neighbourhood_group                          str
neighbourhood                                str
lat                                      float64
long                                     float64
instant_bookable                          object
cancellation_policy                          str
room_type                                    str
construction_year                        float64
price                                    float64
service_fee                              float64
minimum_nights                           float64
number_of_reviews                        float64
last_review                       datetime64[us]
reviews_per_month                        float64
review_rate_number                       float64
calculated_host_listings_count           float64
availability_365    

# Filling with mean and mode , by the help of Ai

In [57]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

mean_cols = ['price', 'service_fee', 'construction_year', 'minimum_nights',
             'number_of_reviews', 'reviews_per_month',
             'calculated_host_listings_count', 'availability_365']

for col in mean_cols:
    df[col] = df[col].fillna(df[col].mean())

mode_cols = ['neighbourhood_group', 'neighbourhood', 'cancellation_policy', 'room_type']

for col in mode_cols:
    df[col] = df[col].fillna(df[col].mode()[0])



# handle outliers 

In [58]:
cols_to_clean = ['price', 'service_fee']

for col in cols_to_clean:
    df[col] = df[col].astype(str).str.replace('$', '', regex=False)
    df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [59]:
cols = ['price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'availability_365']

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

df = df.reset_index(drop=True)
print(df.shape)

(64972, 22)


In [60]:
df.to_csv(r"E:\summit\airbnb_clean.csv", index=False)